# T06 — Structured Outputs

A free-text answer is fine for a chat, but a compliance dashboard or a database needs fields: product, EPD status, GWP value, sources. This notebook makes the RAG agent return JSON that follows a schema, then validates it.

1. Free text and citations in the prompt
2. A JSON schema enforced by the API (structured outputs)
3. Validation of the parsed result
4. Handling questions the documents cannot answer

## Setup

Requires `OPENAI_API_KEY` (locally: in `.env.local` at the repo root). Run T01 first: it builds the vector store the later notebooks reuse.

In [ ]:
import sys

from loguru import logger
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    ask,
    build_agent,
    build_llm,
    load_or_build_vector_store,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
llm = build_llm("openai")
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)

QUERY = "Which products have a third-party verified EPD, and what is their GWP?"
answer = await ask(agent, QUERY)

The facts may be right, but they are buried in prose that another program cannot use.

## 1. Citations via the Prompt

The agent passes each chunk as `<source id='...'>...</source>`. A prompt can ask the model to cite those IDs per fact. Nothing enforces the format, and the IDs are opaque chunk IDs; `ask()` maps them back to files.

In [ ]:
CITING_PROMPT = (
    SYSTEM_PROMPT
    + "\n- After every fact, cite the id of its <source> tag, e.g. (source: 3f2a...)."
)
citing_agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=CITING_PROMPT,
)
answer = await ask(citing_agent, QUERY)

## 2. Structured Outputs

With OpenAI's structured outputs, the API constrains generation to a JSON schema: the response always parses and has every required field. We define the schema as a Pydantic model and pass its JSON schema as `response_format`. `strict` mode requires every field to be listed as required and no extra fields (`extra="forbid"`); optional values are expressed as `float | None`.

In [ ]:
class Product(BaseModel):
    model_config = ConfigDict(extra="forbid")

    product_name: str
    has_verified_epd: bool
    gwp_kg_co2e: float | None = Field(
        description="GWP in kg CO2e per declared unit; null if not stated"
    )
    source_ids: list[str] = Field(description="ids of the <source> tags used")


class ProductCompliance(BaseModel):
    model_config = ConfigDict(extra="forbid")

    products: list[Product]


structured_llm = build_llm(
    "openai",
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "ProductCompliance",
            "strict": True,
            "schema": ProductCompliance.model_json_schema(),
        },
    },
)
structured_agent = build_agent(
    vector_store,
    embedding_model,
    structured_llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)

answer = await ask(structured_agent, QUERY)

## 3. Validation

The schema guarantees structure, not truth. Parse the JSON into the Pydantic model, then check what the schema cannot: plausible values, and source IDs that were actually retrieved (a cited ID that was never retrieved is invented).

In [ ]:
def check(answer) -> ProductCompliance | None:
    try:
        result = ProductCompliance.model_validate_json(answer.content[0].text)
    except ValidationError as error:
        print(f"Invalid response: {error}")
        return None

    retrieved_ids = {source.id for source in answer.sources}
    for product in result.products:
        problems = []
        if product.gwp_kg_co2e is not None and product.gwp_kg_co2e <= 0:
            problems.append("non-positive GWP")
        if product.has_verified_epd and not product.source_ids:
            problems.append("verified EPD claimed without a source")
        if unknown := set(product.source_ids) - retrieved_ids:
            problems.append(f"{len(unknown)} cited id(s) were not retrieved")
        print(f"{product.product_name:<45} {'; '.join(problems) or 'ok'}")
    return result


result = check(answer)

## 4. Questions Without an Answer

When the documents do not contain the answer, the schema still forces a JSON object. The correct output is an empty `products` list, which the calling code must handle explicitly. The model may instead return a similar product (another pallet); the schema cannot catch that, so check that the returned names match what was asked.

In [ ]:
answer = await ask(
    structured_agent, "What is the GWP of the Lara pallet, and does it have an EPD?"
)
result = check(answer)
if result is not None:
    matches = [p for p in result.products if "lara" in p.product_name.lower()]
    others = [p.product_name for p in result.products if p not in matches]
    print(f"Lara pallet entries: {len(matches)}; other products returned: {others}")

## Summary

| Approach | Guarantees | Use when |
|---|---|---|
| Instructions in the prompt | Nothing | Human readers |
| Structured outputs (`response_format`) | Valid JSON matching the schema | Machine consumers |
| Pydantic validation and checks | Types, plausible values, grounded sources | Before storing or displaying results |